# The three routing objectives

PASSAGE solves every scene under three cost objectives. They encode different preferences: stay low
(`elevation`), avoid climbing in proportion to the relief of the crop (`slope_uphill`), and pay for each
metre climbed (`energy`). This notebook shows what each objective rewards on one scene, then measures
on 64 validation scenes how much a route optimal for one objective costs under the others.

For a step from cell $u$ to cell $v$ of length $d_{uv}$ (30 m, or $30\sqrt{2}$ m on a diagonal),
with elevations $h$ in metres, crop extremes $h_{\min}$ and $h_{\max}$, and weight $w$:

$$c_{\mathrm{elevation}} = d_{uv}\left(1 + w\,\frac{h_v - h_{\min}}{h_{\max} - h_{\min}}\right),\quad
c_{\mathrm{slope}} = d_{uv}\left(1 + w\,\frac{\max(0,\,h_v - h_u)}{h_{\max} - h_{\min}}\right),\quad
c_{\mathrm{energy}} = d_{uv} + w\,\max(0,\,h_v - h_u)$$

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It reads 64 preview rows and runs in under a minute.

In [ ]:
resolution = 256
split = "validation"
sample_id = None  # scene of Sections 1 and 2; None takes the first scene whose routes span a quarter of the side
scenes = 64  # preview scenes used in Section 3
output_dir = "notebook-artifacts/notebooks/costmodel"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "65f1e4f0a37d8a7b476ddf02d48fe542542924ea"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))


# The shared trajectory style of the paper: one terrain scale, white start circle and goal star,
# green solid routes on free terrain and blue short dashes when obstacles are blocked.
from trajectory_style import render_cost_comparison, draw_comparison, comparison_handles  # noqa: E402, F401
from passage.plotting import decode_scene, render_scene, terrain_colormap, OBSTACLE_COLOR  # noqa: E402, F401


def first_scene(resolution, split, min_share=0.25, min_coverage=0.05):
    """Selection rule fixed in advance: the first scene of the split whose six stored routes all span at least
    `min_share` of the grid side and whose obstacles cover at least `min_coverage` of the cells."""
    for candidate in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False):
        lengths = [variant["length"] for paths in candidate["metadata"]["paths"].values()
                   for variant in (paths["free"], paths["obstacles"])]
        coverage = candidate["metadata"]["obstacles"]["ratio_actual"]
        if min(lengths) >= min_share * resolution and coverage >= min_coverage:
            return candidate["id"]
    raise ValueError("No scene of the split meets the selection rule")

## 1. Three optimal routes on one terrain

Figure 1 draws the three free-terrain routes of one scene on the same map, coloured by objective.
Figure 2 follows each route from start to goal and plots the elevation it crosses.

In [ ]:
from passage_eval.metrics.cost import path_cost

SOLVERS = {"elevation": "astar_elevation", "slope_uphill": "astar_slope", "energy": "astar_energy"}
row = source.get_row(resolution, split, sample_id=sample_id or first_scene(resolution, split), samples=False)
elevation, obstacles, start, goal = decode_scene(row)
weights = {o: float(row["metadata"]["solvers"][s]["weight"]) for o, s in SOLVERS.items()}
free_routes = {o: np.asarray(row[f"path_{s}_free"]) for o, s in SOLVERS.items()}
fig = render_scene(elevation, None, start, goal, {OBJECTIVE_LABELS[o]: r for o, r in free_routes.items()},
                   title=f"Free-terrain routes · scene {row['id']} · {resolution}×{resolution}")
figures = save_figure(fig, output_path, f"costmodel-{resolution}-{row['id']}-routes")
plt.show()

fig, ax = plt.subplots(figsize=(8.6, 3.4))
fig.subplots_adjust(top=0.78)
for objective, route in free_routes.items():
    steps = np.r_[0, np.hypot(*np.diff(route, axis=0).T)] * 30 / 1000
    ax.plot(np.cumsum(steps), elevation[route[:, 0], route[:, 1]], color=OBJECTIVE_COLORS[objective],
            label=OBJECTIVE_LABELS[objective])
ax.set(xlabel="Distance along the route (km)", ylabel="Elevation (m)")
ax.legend(ncol=3, fontsize=8, loc="best")
figure_title(fig, "Elevation profile of each optimal route", f"Scene {row['id']} · free terrain")
figures += save_figure(fig, output_path, f"costmodel-{resolution}-{row['id']}-profiles")
plt.show()

*Figure 1. The three free-terrain optima of one scene. Figure 2. Elevation along each route against the
distance travelled. The elevation route trades distance for low ground; the slope and energy routes
avoid climbs and often run together. Where they coincide, the energy route (violet) is drawn on top
of the slope route (orange).*

## 2. Each route under each objective

Every route can be priced under every objective. Table 1 divides each price by the optimum of the
pricing objective, so 1.000 marks the optimum and 1.050 a route 5 % more expensive. Exact labels put the
minimum of each column on the diagonal.

In [ ]:
def cross_costs(row, elevation):
    table = {}
    for route_objective, solver in SOLVERS.items():
        route = row[f"path_{solver}_free"]
        table[route_objective] = {
            priced: path_cost(elevation, route, weight=weights[priced], cost_model=priced)
            for priced in SOLVERS}
    frame = pd.DataFrame(table).T  # rows: route optimised for; columns: priced under
    return frame / np.diag(frame.values)[None, :]

ratios = cross_costs(row, elevation)
shown = ratios.rename(index=OBJECTIVE_LABELS, columns=OBJECTIVE_LABELS).reset_index(names="Route optimal for")
show(shown, "Table 1. Cost of each route relative to the optimum of each objective (columns), one scene.", "{:,.3f}")

## 3. The price of the wrong objective, over 64 scenes

The same table, computed on the validation preview scenes, shows how much a route loses when it was
optimised for another objective. Figure 3 reports the median excess cost in percent.

In [ ]:
excess = []
for preview in source.iter_rows(resolution, split, columns=None, samples=True, limit=scenes):
    preview_elevation, _, _, _ = decode_scene(preview)
    ratio = cross_costs(preview, preview_elevation)
    assert np.all(ratio.values >= np.diag(ratio.values)[None, :] - 1e-9), "an off-diagonal route beats the optimum"
    excess.append(100 * (ratio - 1))
median_excess = pd.concat(excess).groupby(level=0).median().loc[list(SOLVERS), list(SOLVERS)]
p90_excess = pd.concat(excess).groupby(level=0).quantile(0.9).loc[list(SOLVERS), list(SOLVERS)]

fig, ax = plt.subplots(figsize=(5.6, 4.4))
fig.subplots_adjust(top=0.8, left=0.26)
image = ax.imshow(median_excess.values, cmap="Blues", vmin=0)
for i in range(3):
    for j in range(3):
        value = median_excess.values[i, j]
        ax.text(j, i, f"{value:.1f} %", ha="center", va="center", fontsize=9,
                color="white" if value > 0.6 * median_excess.values.max() else INK)
ax.set_xticks(range(3), [OBJECTIVE_LABELS[o] for o in SOLVERS])
ax.set_yticks(range(3), [OBJECTIVE_LABELS[o] for o in SOLVERS])
ax.set(xlabel="Priced under", ylabel="Route optimal for")
ax.grid(False)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.03).set_label("Median excess cost (%)")
figure_title(fig, "Price of optimising the wrong objective", f"Median over {len(excess)} validation scenes · free terrain")
figures += save_figure(fig, output_path, f"costmodel-{resolution}-cross-objective")
plt.show()
show((median_excess.round(3).rename(index=OBJECTIVE_LABELS, columns=OBJECTIVE_LABELS)
      .reset_index(names="Route optimal for")), "Table 2. Median excess cost (%), validation preview scenes.", "{:,.2f}")

*Figure 3. Median extra cost of a route optimised for one objective (rows) when priced under another
(columns). The diagonal is zero by construction. A large value means the two objectives disagree about
where to go.*

## Summary

In [ ]:
off = median_excess.where(~np.eye(3, dtype=bool)).stack()
worst = off.idxmax()
display(Markdown("\n".join([
    f"- On all {len(excess)} scenes, every stored route is the cheapest under its own objective: the exact "
    "labels never lose to a route optimised for another objective.",
    f"- The largest median penalty, {off.max():.1f} %, comes from taking the {OBJECTIVE_LABELS[worst[0]].lower()} "
    f"route and pricing it under {OBJECTIVE_LABELS[worst[1]].lower()}.",
    f"- The smallest off-diagonal median, {off.min():.2f} %, comes from pricing the "
    f"{OBJECTIVE_LABELS[off.idxmin()[0]].lower()} route under {OBJECTIVE_LABELS[off.idxmin()[1]].lower()}: "
    "those two objectives often agree.",
])))
report = {"resolution": resolution, "split": split, "scene": row["id"], "scenes": len(excess),
          "one_scene_ratios": ratios.to_dict(), "median_excess_pct": median_excess.to_dict(),
          "p90_excess_pct": p90_excess.to_dict(), "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"costmodel-{resolution}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*